<a href="https://colab.research.google.com/github/novella-tedesco/FLO/blob/Digital-Nomadism-LitRev/LitRevMethodology.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive; drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# =========================
# Abstract methodology classification
# =========================
!pip -q install pandas

import os, re, html, unicodedata, json
import pandas as pd
from collections import Counter

# --- SET THIS to your Drive path with all abstracts (.txt) ---
# Example: "/content/drive/MyDrive/LitRev/abstracts_txt-final"
ABSTRACTS_DIR = "/content/drive/MyDrive/LitRev/abstracts_txt-final"  # <-- change this to your folder
OUT_DIR = os.path.join(ABSTRACTS_DIR, "_method_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# --- Lexicons (heuristics) ---
QUAL = [
    "qualitative","interview","semi-structured","focus group","case study","multiple case",
    "ethnograph","participant observation","netnography","discourse analysis","thematic analysis",
    "grounded theory","narrative analysis","content analysis","coding scheme","fieldwork","observations",
    "interpretive","phenomenolog","autoethnograph"
]
QUANT = [
    "quantitative","survey","questionnaire","regression","anova","ancova","experiment","quasi-experiment",
    "t-test","p-value","statistical analysis","sample size","likert","hypotheses were tested",
    "structural equation","sem","pls-sem","confirmatory factor analysis","cfa","path analysis",
    "logit","probit","panel data","cronbach","alpha reliability"
]
REVIEW = [
    "systematic review","systematic literature review","slr","scoping review","bibliometric","scientometric",
    "meta-analysis","state of the art","review of the literature","literature review"
]
TECH = [
    "policy report","technical report","white paper","prototype","implementation",
    "simulation","architecture design","system design","algorithm","model implementation","roadmap","strategy report"
]

CATEGORIES = [
    "Qualitative research",
    "Quantitative research",
    "Mixed methods",
    "Theoretical or opinion-based",
    "Bibliometric/systematic reviews",
    "Technical studies / reports",
]

def clean_text(t: str) -> str:
    if not t: return ""
    t = html.unescape(t)
    t = unicodedata.normalize("NFKC", t)
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def count_hits(text: str, lexicon):
    t = text.lower()
    hits = []
    for w in lexicon:
        # flexible match: exact word/phrase OR substring for stems like "ethnograph"
        if " " in w or "-" in w:  # phrase/hyphen
            if re.search(r"\b" + re.escape(w.lower()) + r"\b", t):
                hits.append(w)
        else:
            if re.search(r"\b" + re.escape(w.lower()) + r"\w*\b", t):
                hits.append(w)
    return hits

def classify_methodology(text: str):
    """
    Rule-based classification:
    - Reviews first (high precision)
    - Mixed if qual & quant signals co-occur (or explicit 'mixed method')
    - Then qual / quant
    - Then technical
    - Else theoretical/opinion
    """
    t = (text or "").lower()
    hits_review = count_hits(t, REVIEW)
    if hits_review:
        return "Bibliometric/systematic reviews", hits_review

    # explicit mixed method markers
    mixed_markers = []
    if re.search(r"\bmixed[-\s]?method", t) or "triangulation" in t:
        mixed_markers.append("mixed-methods/triangulation")

    hits_qual = count_hits(t, QUAL)
    hits_quant = count_hits(t, QUANT)

    if mixed_markers or (hits_qual and hits_quant):
        return "Mixed methods", (hits_qual + hits_quant + mixed_markers)

    if hits_qual:
        return "Qualitative research", hits_qual

    if hits_quant:
        return "Quantitative research", hits_quant

    hits_tech = count_hits(t, TECH)
    if hits_tech:
        return "Technical studies / reports", hits_tech

    return "Theoretical or opinion-based", []

# ----------- Scan folder -----------
rows = []
n_files = 0
for root, _, files in os.walk(ABSTRACTS_DIR):
    for f in sorted(files):
        if not f.lower().endswith(".txt"):
            continue
        # skip output dir files if scan includes it
        if "_method_outputs" in root:
            continue
        n_files += 1
        path = os.path.join(root, f)
        with open(path, "r", encoding="utf-8", errors="ignore") as fh:
            raw = fh.read()
        txt = clean_text(raw)
        label, markers = classify_methodology(txt)

        # simple confidence: more markers = more confidence; cap at 1.0
        conf = min(1.0, 0.25*len(markers)) if label != "Theoretical or opinion-based" else (0.2 if txt else 0.0)

        rows.append({
            "file": f,
            "path": path,
            "tokens": len(txt.split()),
            "methodology": label,
            "matched_markers": "; ".join(markers),
            "confidence_heuristic": round(conf, 3)
        })

df = pd.DataFrame(rows).sort_values(["methodology","file"])
df.to_csv(os.path.join(OUT_DIR, "methods_by_file.csv"), index=False, encoding="utf-8")

counts = df["methodology"].value_counts().rename_axis("Methodology").reset_index(name="Number of studies")
counts.to_csv(os.path.join(OUT_DIR, "method_counts.csv"), index=False, encoding="utf-8")

print(f"Scanned files: {n_files}")
print("Saved:", os.path.join(OUT_DIR, "methods_by_file.csv"))
print("Saved:", os.path.join(OUT_DIR, "method_counts.csv"))
print(counts.to_string(index=False))


Scanned files: 224
Saved: /content/drive/MyDrive/LitRev/abstracts_txt-final/_method_outputs/methods_by_file.csv
Saved: /content/drive/MyDrive/LitRev/abstracts_txt-final/_method_outputs/method_counts.csv
                    Methodology  Number of studies
   Theoretical or opinion-based                 96
           Qualitative research                 70
                  Mixed methods                 27
Bibliometric/systematic reviews                 15
          Quantitative research                 13
    Technical studies / reports                  3
